In [3]:
# ── Cell 1: Install ──────────────────────────────────────────────
!pip install -U qwen-tts -q
!pip install fastapi uvicorn pyngrok nest-asyncio soundfile -q
# flash-attn removed — using torch SDPA instead (works natively on T4)

import torch
print(f"✅ CUDA available : {torch.cuda.is_available()}")
print(f"✅ GPU name       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

✅ CUDA available : False
✅ GPU name       : CPU only


In [4]:
# ── Cell 2: Load model ───────────────────────────────────────────
import torch
from qwen_tts import Qwen3TTSModel

print("Loading Qwen3-TTS VoiceDesign 1.7B model...")
print("First run downloads ~3.5 GB — subsequent runs load from cache instantly.")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

print("✅ Model loaded successfully!")

# Quick test to confirm everything works
import soundfile as sf
test_wavs, test_sr = model.generate_voice_design(
    text="Model loaded and working correctly.",
    language="English",
    instruct="A calm, professional male voice.",
)
sf.write("/tmp/test_output.wav", test_wavs[0], test_sr)
print("✅ Test generation passed — model is ready.")

Loading Qwen3-TTS VoiceDesign 1.7B model...
First run downloads ~3.5 GB — subsequent runs load from cache instantly.


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# ── Cell 3: FastAPI server ───────────────────────────────────────
import os, uuid, base64, nest_asyncio, io
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import soundfile as sf
import numpy as np
import uvicorn

nest_asyncio.apply()

PLATFORM = "colab"

app = FastAPI(title=f"Qwen3-TTS API — {PLATFORM.upper()}")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Request / Response schemas ────────────────────────────────────

class GenerateRequest(BaseModel):
    text: str                       # Text to narrate
    instruct: str                   # Voice style instruction
    language: str = "English"       # English / Chinese / Japanese / Korean / German /
                                    # French / Russian / Portuguese / Spanish / Italian

class GenerateResponse(BaseModel):
    audio_b64: str                  # Base64-encoded WAV
    sample_rate: int
    platform: str
    duration_seconds: float

# ── Supported languages ────────────────────────────────────────────
SUPPORTED_LANGUAGES = [
    "English", "Chinese", "Japanese", "Korean",
    "German", "French", "Russian", "Portuguese",
    "Spanish", "Italian"
]

# ── Health check endpoint ─────────────────────────────────────────
@app.get("/")
def health():
    return {
        "status": "ok",
        "platform": PLATFORM,
        "model": "Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "supported_languages": SUPPORTED_LANGUAGES,
    }

# ── Main generation endpoint ──────────────────────────────────────
@app.post("/generate", response_model=GenerateResponse)
def generate_voice(req: GenerateRequest):

    # Validate
    if not req.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty.")
    if len(req.text) > 500:
        raise HTTPException(status_code=400, detail="Text too long. Max 500 characters.")
    if not req.instruct.strip():
        raise HTTPException(status_code=400, detail="Instruct cannot be empty.")
    if len(req.instruct) > 300:
        raise HTTPException(status_code=400, detail="Instruct too long. Max 300 characters.")
    if req.language not in SUPPORTED_LANGUAGES:
        raise HTTPException(
            status_code=400,
            detail=f"Unsupported language. Choose from: {', '.join(SUPPORTED_LANGUAGES)}"
        )

    try:
        # Generate audio
        wavs, sr = model.generate_voice_design(
            text=req.text,
            language=req.language,
            instruct=req.instruct,
        )

        audio_array = wavs[0]
        duration = len(audio_array) / sr

        # Write to in-memory buffer
        buf = io.BytesIO()
        sf.write(buf, audio_array, sr, format="WAV")
        buf.seek(0)
        audio_b64 = base64.b64encode(buf.read()).decode("utf-8")

        return GenerateResponse(
            audio_b64=audio_b64,
            sample_rate=sr,
            platform=PLATFORM,
            duration_seconds=round(duration, 2),
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Generation failed: {str(e)}")

In [ ]:
# ── Cell 4: Start Ngrok + Uvicorn ────────────────────────────────
from pyngrok import ngrok
from google.colab import userdata
import uvicorn, asyncio

# Load Ngrok token from Colab Secrets
ngrok_token = userdata.get("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)
ngrok.kill()

tunnel = ngrok.connect(8000, bind_tls=True)

print("=" * 60)
print(f"  ✅ COLAB API URL  : {tunnel.public_url}")
print(f"  🔍 Health Check   : {tunnel.public_url}/")
print(f"  🎙️  Generate Voice : {tunnel.public_url}/generate")
print("=" * 60)
print("  ⚠️  Copy this URL into your frontend SERVERS config!")
print("=" * 60)

# Keep-alive heartbeat
import threading, time
def heartbeat():
    while True:
        time.sleep(3000)
        print("♥ Colab heartbeat")
threading.Thread(target=heartbeat, daemon=True).start()

# ✅ Fix: use async serve instead of uvicorn.run()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()